# MVC Project: Multilayer Perceptron
## Task 7 - Python Implementation
**Artificial Neural Networks**

This notebook implements the full MLP pipeline from Tasks 1–6 using NumPy only, trained on the MNIST dataset.

## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import os

np.random.seed(42)
print('Libraries loaded successfully.')

## Load MNIST Dataset

In [ ]:
# Download MNIST if not already present
mnist_path = 'mnist2.npz'
if not os.path.exists(mnist_path):
    print('Downloading MNIST...')
    url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
    urllib.request.urlretrieve(url, mnist_path)
    print('Downloaded.')

data = np.load(mnist_path)
X_train = data['x_train'].reshape(-1, 784) / 255.0   # (60000, 784)
Y_train = data['y_train']                             # (60000,)
X_test  = data['x_test'].reshape(-1, 784) / 255.0    # (10000, 784)
Y_test  = data['y_test']                             # (10000,)

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')

# One-hot encode labels
def one_hot(y, num_classes=10):
    m = y.shape[0]
    oh = np.zeros((m, num_classes))
    oh[np.arange(m), y] = 1
    return oh

Y_train_oh = one_hot(Y_train)  # (60000, 10)
Y_test_oh  = one_hot(Y_test)   # (10000, 10)
print(f'Y_train_oh shape: {Y_train_oh.shape}')

## Network Architecture

| Layer | Type | Neurons | Activation |
|-------|------|---------|------------|
| 0 | Input | 784 | - |
| 1 | Hidden 1 | 128 | ReLU |
| 2 | Hidden 2 | 64 | ReLU |
| 3 | Output | 10 | Softmax |

In [ ]:
# Weight initialisation: uniform random in [-0.5, 0.5], biases = 0
np.random.seed(42)

W1 = np.random.uniform(-0.5, 0.5, (784, 128))   # Input -> Hidden1
b1 = np.zeros((1, 128))

W2 = np.random.uniform(-0.5, 0.5, (128, 64))    # Hidden1 -> Hidden2
b2 = np.zeros((1, 64))

W3 = np.random.uniform(-0.5, 0.5, (64, 10))     # Hidden2 -> Output
b3 = np.zeros((1, 10))

print('Weights initialised.')
print(f'W1: {W1.shape}, W2: {W2.shape}, W3: {W3.shape}')

## Step 1 – Sigmoid Activation (Task 1 / Task 3)

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(a):
    """Derivative of ReLU expressed in terms of its output a.
    1 where a > 0, else 0.
    """
    return (a > 0).astype(float)

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

def softmax(z):
    """Numerically stable softmax (subtract row max before exp)."""
    e = np.exp(z - np.max(z, axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

print('relu(0)    =', relu(0))           # 0
print('relu(2)    =', relu(2))           # 2
print('relu(-1)   =', relu(-1))          # 0


## Step 2 – Forward Pass (Task 1)

In [ ]:
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    """Compute a full forward pass through the 3-layer MLP.
    
    Parameters
    ----------
    X   : (m, 784) input batch
    W1,b1 : Layer 1 weights and biases
    W2,b2 : Layer 2 weights and biases
    W3,b3 : Output layer weights and biases

    Returns
    -------
    Z1,A1 : pre-activation and activation of Layer 1
    Z2,A2 : pre-activation and activation of Layer 2
    Z3,A3 : pre-activation and activation of Output layer
    """
    # Layer 1: Hidden Layer 1
    Z1 = X @ W1 + b1           # (m, 128)
    A1 = relu(Z1)              # was sigmoid

    # Layer 2: Hidden Layer 2
    Z2 = A1 @ W2 + b2          # (m, 64)
    A2 = relu(Z2)              # was sigmoid

    # Output Layer
    Z3 = A2 @ W3 + b3          # (m, 10)
    A3 = softmax(Z3)           # was sigmoid

    return Z1, A1, Z2, A2, Z3, A3

# Sanity check on a single sample
_,_,_,_,_,A3_check = forward_pass(X_train[:1], W1, b1, W2, b2, W3, b3)
print('Output for first training sample (should sum ~1 for probabilities):')
print(A3_check)

## Step 3 – MSE Loss (Task 2)

In [ ]:
def cross_entropy_loss(Y_true, Y_pred):
    """Categorical cross-entropy.
    L = -(1/m) * sum(Y_true * log(Y_pred))
    Clipped to avoid log(0).
    """
    return -np.mean(np.sum(Y_true * np.log(np.clip(Y_pred, 1e-8, 1.0)), axis=1))

np.random.seed(42)
W1 = np.random.uniform(-0.5, 0.5, (784, 128)); b1 = np.zeros((1, 128))
W2 = np.random.uniform(-0.5, 0.5, (128, 64));  b2 = np.zeros((1, 64))
W3 = np.random.uniform(-0.5, 0.5, (64, 10));   b3 = np.zeros((1, 10))

_,_,_,_,_,A3_init = forward_pass(X_train[:1000], W1, b1, W2, b2, W3, b3)
print('Initial CE loss:', cross_entropy_loss(Y_train_oh[:1000], A3_init))

## Step 4 – Backpropagation (Task 3)

In [ ]:
def backpropagation(X, Y_true, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3):
    m = X.shape[0]

    # Output delta: softmax + cross-entropy derivative collapses to (A3 - Y) / m
    # No sigmoid_derivative needed here — the cancellation is exact.
    delta3 = (A3 - Y_true) / m                             # (m, 10)

    dW3 = A2.T @ delta3
    db3 = np.sum(delta3, axis=0, keepdims=True)

    # Hidden layer 2 delta — use relu_derivative instead of sigmoid_derivative
    delta2 = (delta3 @ W3.T) * relu_derivative(A2)         # (m, 64)

    dW2 = A1.T @ delta2
    db2 = np.sum(delta2, axis=0, keepdims=True)

    # Hidden layer 1 delta
    delta1 = (delta2 @ W2.T) * relu_derivative(A1)         # (m, 128)

    dW1 = X.T @ delta1
    db1 = np.sum(delta1, axis=0, keepdims=True)

    return dW1, db1, dW2, db2, dW3, db3

print('backpropagation() defined.')

## Step 5 – Weight Update (Task 4)

In [ ]:
def update_weights(W1, b1, W2, b2, W3, b3,
                   dW1, db1, dW2, db2, dW3, db3,
                   learning_rate):
    """Apply gradient descent update rule: W = W - lr * dW."""
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1

    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

    W3 = W3 - learning_rate * dW3
    b3 = b3 - learning_rate * db3

    return W1, b1, W2, b2, W3, b3

print('update_weights() defined.')

## Training Loop – Mini-Batch Gradient Descent (Task 5)

Batch size = 32, η = 0.1, 20 epochs.

In [ ]:
np.random.seed(42)
W1 = np.random.uniform(-0.5, 0.5, (784, 128)); b1 = np.zeros((1, 128))
W2 = np.random.uniform(-0.5, 0.5, (128, 64));  b2 = np.zeros((1, 64))
W3 = np.random.uniform(-0.5, 0.5, (64, 10));   b3 = np.zeros((1, 10))

learning_rate = 0.1
epochs        = 40
batch_size    = 32
loss_history  = []

for epoch in range(epochs):
    idx    = np.random.permutation(X_train.shape[0])
    X_shuf = X_train[idx]
    Y_shuf = Y_train_oh[idx]

    for start in range(0, X_train.shape[0], batch_size):
        X_b = X_shuf[start:start+batch_size]
        Y_b = Y_shuf[start:start+batch_size]

        Z1,A1,Z2,A2,Z3,A3 = forward_pass(X_b, W1,b1,W2,b2,W3,b3)
        grads = backpropagation(X_b,Y_b, Z1,A1,Z2,A2,Z3,A3, W1,W2,W3)
        W1,b1,W2,b2,W3,b3 = update_weights(W1,b1,W2,b2,W3,b3, *grads, learning_rate)

    _,_,_,_,_,A3_full = forward_pass(X_train, W1,b1,W2,b2,W3,b3)
    epoch_loss = cross_entropy_loss(Y_train_oh, A3_full)   # CE, not MSE
    loss_history.append(epoch_loss)
    print(f'Epoch {epoch+1:2d}/{epochs}  |  CE Loss: {epoch_loss:.4f}')

print('Training complete.')

## Output 1 – Loss Curve

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), loss_history, marker='o', color='steelblue', linewidth=2)
plt.title('Cross-Entropy Loss vs. Epoch (MNIST Training)', fontsize=14)
plt.xlabel('Epoch')
plt.ylabel('CE Loss')
plt.xticks(range(1, epochs + 1))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()
print(f'Final training loss: {loss_history[-1]:.4f}')

## Output 2 – Test Accuracy

In [ ]:
_, _, _, _, _, A3_test = forward_pass(X_test, W1, b1, W2, b2, W3, b3)

# Predicted class = index of highest output neuron
Y_pred_classes = np.argmax(A3_test, axis=1)   # (10000,)

accuracy = np.mean(Y_pred_classes == Y_test) * 100
print(f'Test Accuracy: {accuracy:.2f}%')

## Output 3 – Sample Predictions (one per digit class)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample Predictions – One per Digit Class', fontsize=14)

for digit in range(10):
    # Find first test image whose TRUE label matches this digit
    idx = np.where(Y_test == digit)[0][0]
    image       = X_test[idx].reshape(28, 28)
    true_label  = Y_test[idx]
    pred_label  = Y_pred_classes[idx]

    ax = axes[digit // 5][digit % 5]
    ax.imshow(image, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'True: {true_label}  |  Pred: {pred_label}', fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150)
plt.show()

## Expansion

In [21]:
## Export Trained Weights
import json

print("W1 shape:", W1.shape)   # should be (784, 128)
print("W2 shape:", W2.shape)   # should be (128, 64)
print("W3 shape:", W3.shape)   # should be (64, 10)

weights = {
    "W1": W1.tolist(), "b1": b1.tolist(),
    "W2": W2.tolist(), "b2": b2.tolist(),
    "W3": W3.tolist(), "b3": b3.tolist()
}
with open("weights.json", "w") as f:
    json.dump(weights, f)
print("Weights saved to weights.json")

---
## (Optional) Gradient Verification with PyTorch

Use PyTorch **only** to verify that your NumPy gradients are correct — not for training.

In [34]:
try:
    import torch
    import torch.nn as nn

    # Use a tiny batch for verification
    X_v = torch.tensor(X_train[:4], dtype=torch.float32)
    Y_v = torch.tensor(Y_train_oh[:4], dtype=torch.float32)

    W1_t = torch.tensor(W1, dtype=torch.float32, requires_grad=True)
    b1_t = torch.tensor(b1, dtype=torch.float32, requires_grad=True)
    W2_t = torch.tensor(W2, dtype=torch.float32, requires_grad=True)
    b2_t = torch.tensor(b2, dtype=torch.float32, requires_grad=True)
    W3_t = torch.tensor(W3, dtype=torch.float32, requires_grad=True)
    b3_t = torch.tensor(b3, dtype=torch.float32, requires_grad=True)

    # Use ReLU + logits + CrossEntropy for verification
    A1_t = torch.relu(X_v @ W1_t + b1_t)
    A2_t = torch.relu(A1_t @ W2_t + b2_t)
    Z3_t = A2_t @ W3_t + b3_t  # logits

    labels = torch.tensor(Y_train[:4], dtype=torch.long)
    loss_fn = nn.CrossEntropyLoss()
    loss_t = loss_fn(Z3_t, labels)
    loss_t.backward()

    # Compare dW3
    Z1_v, A1_v, Z2_v, A2_v, Z3_v, A3_v = forward_pass(
        X_train[:4], W1, b1, W2, b2, W3, b3)
    dW1_np, _, dW2_np, _, dW3_np, _ = backpropagation(
        X_train[:4], Y_train_oh[:4],
        Z1_v, A1_v, Z2_v, A2_v, Z3_v, A3_v, W1, W2, W3)

    max_diff = np.max(np.abs(dW3_np - W3_t.grad.numpy()))
    print(f'Max absolute difference in dW3 (NumPy vs PyTorch): {max_diff:.6f}')
    if max_diff < 1e-4:
        print('Gradients MATCH — backpropagation is correct!')
    else:
        print('WARNING: Gradient mismatch — review backpropagation.')
except ImportError:
    print('PyTorch not installed — skipping gradient verification.')